# Extraction des données pour la fusion

In [2]:
import os
import requests
import pandas as pd
import gzip
import time
from io import BytesIO

## Réccupération automatique des données Météo,

Le problème de météo france est le fait qu'a l'heure actuel on doit téléchargé les mois 1 par 1.

In [4]:
output_folder = "climat_mensuel"
os.makedirs(output_folder, exist_ok=True)

start_year = 1990
end_year = 2025

base_url = "https://donneespubliques.meteofrance.fr/donnees_libres/Txt/Climat/climat.{date}.csv.gz"

for year in range(start_year, end_year + 1):
    for month in range(1, 13):
        date_str = f"{year}{month:02d}"
        url = base_url.format(date=date_str)
        local_file = os.path.join(output_folder, f"climat.{date_str}.csv.gz")

        if os.path.exists(local_file):
            print(f"[SKIP] {local_file} existe déjà")
            continue

        try:
            r = requests.get(url, timeout=10)
            if r.status_code == 200:
                with open(local_file, "wb") as f:
                    f.write(r.content)
                print(f"[OK] Téléchargé {date_str}")
            else:
                print(f"[MANQUANT] {date_str} (HTTP {r.status_code})")
        except Exception as e:
            print(f"[ERREUR] {date_str} : {e}")

print("Téléchargement terminé !")

[SKIP] climat_mensuel\climat.199001.csv.gz existe déjà
[SKIP] climat_mensuel\climat.199002.csv.gz existe déjà
[SKIP] climat_mensuel\climat.199003.csv.gz existe déjà
[SKIP] climat_mensuel\climat.199004.csv.gz existe déjà
[SKIP] climat_mensuel\climat.199005.csv.gz existe déjà
[SKIP] climat_mensuel\climat.199006.csv.gz existe déjà
[SKIP] climat_mensuel\climat.199007.csv.gz existe déjà
[SKIP] climat_mensuel\climat.199008.csv.gz existe déjà
[SKIP] climat_mensuel\climat.199009.csv.gz existe déjà
[SKIP] climat_mensuel\climat.199010.csv.gz existe déjà
[SKIP] climat_mensuel\climat.199011.csv.gz existe déjà
[SKIP] climat_mensuel\climat.199012.csv.gz existe déjà
[SKIP] climat_mensuel\climat.199101.csv.gz existe déjà
[SKIP] climat_mensuel\climat.199102.csv.gz existe déjà
[SKIP] climat_mensuel\climat.199103.csv.gz existe déjà
[SKIP] climat_mensuel\climat.199104.csv.gz existe déjà
[SKIP] climat_mensuel\climat.199105.csv.gz existe déjà
[SKIP] climat_mensuel\climat.199106.csv.gz existe déjà
[SKIP] cli

In [5]:
input_folder = "climat_mensuel"
output_file = "../data/in/meteo.csv"

all_files = [f for f in os.listdir(input_folder) if f.endswith(".csv.gz")]
all_files.sort()
dfs = []

for file in all_files:
    file_path = os.path.join(input_folder, file)
    
    with gzip.open(file_path, 'rt', encoding='utf-8') as f:
        df = pd.read_csv(f, sep=";")
        dfs.append(df)

meteo_complet = pd.concat(dfs, ignore_index=True)

meteo_complet.to_csv(output_file, sep=";", index=False)

print(f"[OK] {len(all_files)} fichiers fusionnés dans {output_file}")

[OK] 432 fichiers fusionnés dans ../data/in/meteo.csv


## Réccupération automatique des données Météos 2

In [20]:
dataset_id = "6569b3d7d193b4daf2b43edc"  # climat mensuel

# Dossier
os.makedirs("climat_45", exist_ok=True)

# ==========================================
# 1. Récupérer les ressources du dataset
# ==========================================
url = f"https://www.data.gouv.fr/api/1/datasets/{dataset_id}"
r = requests.get(url)
data = r.json()
resources = data["resources"]

# ==========================================
# 2. Trouver tous les fichiers du département 45
# ==========================================
urls_45 = []

for res in resources:
    name = (res["title"] + res["url"]).lower()
    if "_45" in name:
        urls_45.append(res["url"])
        print("Trouvé :", res["title"])

if not urls_45:
    print("Aucun fichier pour le département 45")
    exit()

print(f"{len(urls_45)} fichiers trouvés")

# ==========================================
# 3. Télécharger + lire tous les fichiers
# ==========================================
all_dfs = []

for i, file_url in enumerate(urls_45):
    print(f"Téléchargement {i+1}/{len(urls_45)}")

    local_file = f"climat_45/tmp_{i}.csv.gz"
    r = requests.get(file_url)

    with open(local_file, "wb") as f:
        f.write(r.content)

    # Lecture (pandas gère gzip automatiquement)
    df = pd.read_csv(local_file, sep=";", compression="gzip")
    all_dfs.append(df)

# ==========================================
# 4. Fusion en un seul DataFrame
# ==========================================
df_final = pd.concat(all_dfs, ignore_index=True)

# Optionnel : tri par station et date
if "NUM_POSTE" in df_final.columns and "AAAAMM" in df_final.columns:
    df_final = df_final.sort_values(["NUM_POSTE", "AAAAMM"])

# ==========================================
# 5. Sauvegarde finale
# ==========================================
output_path = "climat_45/climat_mensuel_45_total.csv"
df_final.to_csv(output_path, sep=";", index=False)

print("Fichier final créé :", output_path)
print("Nombre total de lignes :", len(df_final))

Trouvé : MENS_departement_45_periode_1871-1949
Trouvé : MENS_departement_45_periode_1950-2024
Trouvé : MENS_departement_45_periode_2025-2026
3 fichiers trouvés
Téléchargement 1/3
Téléchargement 2/3
Téléchargement 3/3
Fichier final créé : climat_45/climat_mensuel_45_total.csv
Nombre total de lignes : 48218


## Réccupération automatique des fichiers du niveau de l'eau des nappes

In [4]:
departement = "45"
output_folder = "nappe_piezometres"
os.makedirs(output_folder, exist_ok=True)

url_stations = "https://hubeau.eaufrance.fr/api/v1/niveaux_nappes/stations"

codes = []
page = 1

while True:
    params = {
        "code_departement": departement,
        "size": 200,
        "page": page
    }
    
    r = requests.get(url_stations, params=params)
    data = r.json()["data"]
    
    if not data:
        break
    
    codes.extend([(s["code_bss"],s["x"],s["y"]) for s in data])
    print(f"Page {page} : {len(data)} stations")
    page += 1

codes = list(set(codes))
print(f"Total stations trouvées : {len(codes)}")

url_chroniques = "https://hubeau.eaufrance.fr/api/v1/niveaux_nappes/chroniques"

for i, (code, lat, lon) in enumerate(codes):
    params = {
        "code_bss": code,
        "size": 20000
    }
    
    try:
        r = requests.get(url_chroniques, params=params)
        data = r.json()["data"]
        
        if not data:
            print(f"[{i+1}/{len(codes)}] {code} : aucune donnée")
            continue
        
        df = pd.DataFrame(data)
        df["lat"] = lat
        df["lon"] = lon
        
        safe_code = code.replace("/", "_")
        file_path = os.path.join(output_folder, f"nappe_{safe_code}.csv")
        
        df.to_csv(file_path, sep=";", index=False)
        print(f"[{i+1}/{len(codes)}] {code} : {len(df)} lignes")
        
        time.sleep(0.2)
        
    except Exception as e:
        print(f"[{i+1}/{len(codes)}] Erreur {code} : {e}")

print("Téléchargement terminé")

Page 1 : 120 stations
Total stations trouvées : 120
[1/120] 03278X0020/P : 9300 lignes
[2/120] BSS004LLWY/X : 250 lignes
[3/120] 03636X0385/P : 2015 lignes
[4/120] 04004X0007/F : 10803 lignes
[5/120] 03646X0086/F1 : 8702 lignes
[6/120] 04001X0113/F : 248 lignes
[7/120] 02928X1014/P : 9488 lignes
[8/120] 03631X0099/F : 10575 lignes
[9/120] 03276X0057/P : 1624 lignes
[10/120] 03638X0018/P : 1379 lignes
[11/120] 04323X0005/F : 9937 lignes
[12/120] 03277X0019/S1 : 134 lignes
[13/120] BSS003MEAA/X : 1171 lignes
[14/120] 03273X0022/S1 : 1999 lignes
[15/120] 03652X0032/P : aucune donnée
[16/120] 03986X0028/P1 : 302 lignes
[17/120] 03642X0072/F : 190 lignes
[18/120] 03646X0083/F : 534 lignes
[19/120] 03656X0017/PF : 226 lignes
[20/120] 03273X0026/P : 557 lignes
[21/120] 03636X1060/PZ2 : 3135 lignes
[22/120] 04324X0011/F : 2382 lignes
[23/120] 03651X0043/S1 : 36 lignes
[24/120] 03635X0545/PZ1 : 3921 lignes
[25/120] 03282X0043/S1 : 12890 lignes
[26/120] 03288X0096/F : 247 lignes
[27/120] 03991X0

## Réccupération automatique des fichiers d'ETP

In [22]:
import requests
import os

dataset_id = "667eae35510cd549fc7722c1"
output_folder = "etp_fao_hargreaves"
os.makedirs(output_folder, exist_ok=True)

# Récupérer la liste des ressources
resp = requests.get(f"https://www.data.gouv.fr/api/1/datasets/{dataset_id}/")
data = resp.json()

for resource in data.get("resources", []):
    url = resource.get("url")
    fmt = resource.get("format", "").lower()
    if fmt in ["csv", "csv.gz"]:
        filename = os.path.join(output_folder, url.split("/")[-1])
        print(f"[DOWNLOAD] {url}")
        r = requests.get(url)
        with open(filename, "wb") as f:
            f.write(r.content)

print("Téléchargement ETP FAO Hargreaves terminé !")


[DOWNLOAD] https://object.files.data.gouv.fr/meteofrance/data/synchro_ftp/REF_CC/ETP_FAO/ETP_Hargreaves_coefficient_0.175_1970-1979.csv.gz
[DOWNLOAD] https://object.files.data.gouv.fr/meteofrance/data/synchro_ftp/REF_CC/ETP_FAO/ETP_Hargreaves_coefficient_0.175_1980-1989.csv.gz
[DOWNLOAD] https://object.files.data.gouv.fr/meteofrance/data/synchro_ftp/REF_CC/ETP_FAO/ETP_Hargreaves_coefficient_0.175_1990-1999.csv.gz
[DOWNLOAD] https://object.files.data.gouv.fr/meteofrance/data/synchro_ftp/REF_CC/ETP_FAO/ETP_Hargreaves_coefficient_0.175_2000-2009.csv.gz
[DOWNLOAD] https://object.files.data.gouv.fr/meteofrance/data/synchro_ftp/REF_CC/ETP_FAO/ETP_Hargreaves_coefficient_0.175_2010-2019.csv.gz
[DOWNLOAD] https://object.files.data.gouv.fr/meteofrance/data/synchro_ftp/REF_CC/ETP_FAO/ETP_Hargreaves_coefficient_0.175_latest-2020-2024.csv.gz
Téléchargement ETP FAO Hargreaves terminé !
Lecture des fichiers...
Lecture : ETP_Hargreaves_coefficient_0.175_1970-1979.csv.gz


BadGzipFile: Not a gzipped file (b'PK')

In [12]:
import pandas as pd
import os
import zipfile

folder = "etp_fao_hargreaves"

all_dfs = []

print("Lecture des fichiers...")
maille_file = "maille.csv"  # ton CSV avec les colonnes lat_dg, lon_dg, num_maille
maille_df = pd.read_csv(maille_file, sep=";")

# Bornes du Loiret
LAT_MIN, LAT_MAX = 47.416069, 48.392186
LON_MIN, LON_MAX = 1.474425, 3.221622

# Garder uniquement les mailles dans le Loiret
maille_loiret = maille_df[
    (maille_df["lat_dg"] >= LAT_MIN) & (maille_df["lat_dg"] <= LAT_MAX) &
    (maille_df["lon_dg"] >= LON_MIN) & (maille_df["lon_dg"] <= LON_MAX)
]

maille_loiret_xy = maille_loiret[["lambx", "lamby", "lat_dg", "lon_dg",]]

print("Nombre de mailles Loiret :", len(maille_loiret))
mailles_loiret = set(maille_loiret["num_maille"])

for file in os.listdir(folder):
    path = os.path.join(folder, file)
    
    if file.endswith(".csv"):
        df = pd.read_csv(path, sep=";") 
    elif file.endswith(".csv.gz"):
        print(path)
        try:
            with zipfile.ZipFile(path, 'r') as z:
                for name in z.namelist():
                    if name.endswith(".csv"):
                        with z.open(name) as f:
                            df = pd.read_csv(f, sep=";")
        except :
            pass
        
    

    if "NUM_MAILLE" in df.columns:
        df = df[df["NUM_MAILLE"].isin(mailles_loiret)]
    elif "LAMBX" in df.columns and "LAMBY" in df.columns:
        df = df.merge(
            maille_loiret_xy,
            left_on=["LAMBX", "LAMBY"],
            right_on=["lambx", "lamby"],
            how="inner"
        )
    
    all_dfs.append(df)

# =====================================
# Fusion
# =====================================
if all_dfs:
    etp_all = pd.concat(all_dfs, ignore_index=True)
    etp_all = etp_all[["DATE","ETP_Q_H0175","lat_dg","lon_dg"]]
    
    # Tri optionnel (recommandé)
    if "DATE" in etp_all.columns:
        etp_all = etp_all.sort_values("DATE")
    
    # Sauvegarde
    output_file = os.path.join(folder, "etp_france_total.csv")
    etp_all.to_csv(output_file, sep=";", index=False)
    
    print("Fichier final créé :", output_file)
    print("Nombre de lignes :", len(etp_all))
else:
    print("Aucun fichier trouvé")

Lecture des fichiers...
Nombre de mailles Loiret : 224
etp_fao_hargreaves\ETP_Hargreaves_coefficient_0.175_1970-1979.csv.gz
etp_fao_hargreaves\ETP_Hargreaves_coefficient_0.175_1980-1989.csv.gz
etp_fao_hargreaves\ETP_Hargreaves_coefficient_0.175_1990-1999.csv.gz
etp_fao_hargreaves\ETP_Hargreaves_coefficient_0.175_2000-2009.csv.gz
etp_fao_hargreaves\ETP_Hargreaves_coefficient_0.175_2010-2019.csv.gz
Fichier final créé : etp_fao_hargreaves\etp_france_total.csv
Nombre de lignes : 4090688


## Réccupération automatique des fichiers lien entre etp et géoloc